In [5]:
%pip install -q langchain langchain-community langchain-core langgraph langchain-google-genai langchain-text-splitters faiss-cpu python-dotenv pypdf sentence-transformers


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.embeddings.fake import FakeEmbeddings
from huggingface_hub import snapshot_download

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition

# Load environment variables from .env (if present)
load_dotenv()

# Initialize free HuggingFace embeddings (runs locally)
# If the environment can't reach Hugging Face, fall back to FakeEmbeddings to avoid crashing.
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

try:
    model_path = snapshot_download(
        repo_id=MODEL_ID,
        token=os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    )
    embeddings = HuggingFaceEmbeddings(
        model_name=model_path,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
except OSError as e:
    print(f"WARNING: Could not download/load {MODEL_ID}. Falling back to FakeEmbeddings.\n{e}")
    embeddings = FakeEmbeddings(size=384)

google_api_key = os.getenv("GOOGLE_API_KEY")


In [11]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=google_api_key)

In [15]:
loader=PyPDFLoader("sample1.pdf")
documents = loader.load()

In [16]:
len(documents)

1

In [17]:
spillter = RecursiveCharacterTextSplitter(
    chunk_size=10,chunk_overlap=0
)
chunks = spillter.split_documents(documents)

In [18]:
len(chunks)

24

In [20]:
vectorstore = FAISS.from_documents(chunks, embeddings)

In [21]:
vectorstore

In [22]:
retreiver = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":2})

In [24]:
@tool
def rag_tool(query):

    """
    Retrieve relevant information from the pdf document.
    Use this tool when the user asks factual / conceptual questions
    that might be answered from the stored documents.
    """
    result = retreiver.invoke(query)

    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query': query,
        'context': context,
        'metadata': metadata
    }

In [25]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [26]:
class ChatState(TypedDict):
  message: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):

    messages = state['messages']

    response = llm_with_tools.invoke(messages)

    return {'messages': [response]}